# Light Text Cleaning Only

**Scope:** This notebook handles ONLY the shared preprocessing that both
downstream teams (Sentiment + Topic Modelling) need. It does NOT tokenize,
lemmatize, remove stopwords, or create n-grams — those are for later steps before model creation

**Input:** Folder of MDA `.txt` files named like `Alphabet Inc._10-Q_2025-09-01_MDA.txt`

**Output:** Single DataFrame `df_shared` with:

- `doc_id`, `company_name`, `filing_type`, `filing_date`, `year`, `quarter`
- `sentences` — list of clean sentences (→ Sentiment Analysis team)
- `clean_text` — full cleaned text rejoined (→ Topic Analysis team)
- `n_sentences` — sentence count for QA

**Pipeline Steps:**

1. **Section-level removal** — FLS / safe harbor, tables, headers, boilerplate paragraphs
2. **Text cleaning** — lowercase, strip `$/%`, normalize whitespace
3. **Sentence segmentation** — split + drop fragments < 10 words


## Imports & Config


In [1]:
import re
import pandas as pd
import nltk
from pathlib import Path
from nltk.tokenize import sent_tokenize
from joblib import Parallel, delayed

nltk.download("punkt_tab", quiet=True)

# ── Config ──────────────────────────────────────────────────────────
MDA_FOLDER = "datasets/raw_mda/"
FILE_PATTERN = "*_MDA.txt"
MIN_SENT_WORDS = 10  # drop sentences shorter than this
N_JOBS = -1  # -1 = use all CPU cores


## Step 1 — Section-Level Removal Patterns

Three categories of content to remove before any text analysis:

1. **Forward-looking statements (FLS)** — safe harbor boilerplate at the start
2. **Tables & numeric blocks** — financial statement tables, data rows, orphan numbers
3. **Boilerplate paragraphs** — accounting policy definitions, off-balance sheet disclaimers, etc.


In [2]:
# ── 1A. Forward-looking statement (FLS) boundary ───────────────────────────
# Everything before the first substantive heading is safe-harbor boilerplate.
FLS_END_ANCHORS = [
    r"Overview\s+Our\s+Company\s+and\s+Our\s+Businesses",
    r"Overview\s+Our\s+Company",
    r"Business\s+Overview\s",
    r"Company\s+Overview\s",
    r"Corporate\s+Overview\s",
    r"General\s+Overview\s",
    r"Overview\s+of\s+the\s+Company\s",
    r"Overview\s+of\s+Our\s+Business\s",
    r"Overview\s",  # catches all remaining Overview variants (IGNORECASE)
    r"Executive\s+Summary\s",
    r"Executive\s+Overview\s",
    r"Management\s+Summary\s",
]
_FLS_PATTERN = re.compile("|".join(FLS_END_ANCHORS), re.IGNORECASE)


# ── 1B. Table & numeric block patterns ─────────────────────────────────────
_VAL = (
    r"(?:\$\s*)?-?[\d,]+(?:\.\d+)?\s*(?:%|pts)?"
    r"|\([0-9,\.]+\)\s*(?:%|pts)?"
)
_DATE = (
    r"(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)"
    r"\s+\d{1,2},?\s*\d{4}"
)
_PERIOD = (
    r"Three\s+Months\s+Ended"
    r"|Six\s+Months\s+Ended"
    r"|Nine\s+Months\s+Ended"
    r"|Twelve\s+Months\s+Ended"
    r"|Year\s+Ended"
    r"|Fiscal\s+Year\s+Ended"
    r"|Quarter-over-Quarter\s+Change"
    r"|Year-over-Year\s+Change"
    r"|\$\s*Change"
    r"|%\s*Change"
)
_UNITS = r"\(\$?\s*[Ii]n\s+(?:thousand|million|billion)[^)]*\)"

RE_HEADER_ROW = re.compile(
    r"(?:{date}|{period}|{units})"
    r"(?:\s+(?:{date}|{period}|{units}))*".format(
        date=_DATE, period=_PERIOD, units=_UNITS
    ),
    re.IGNORECASE,
)
RE_DATA_ROW = re.compile(
    r"(?<![.!?]\s)"
    r"[A-Za-z][A-Za-z0-9 \&\(\),\-\.]{{1,50}}"
    r"(?:\s+(?:{val})){{2,}}".format(val=_VAL),
    re.IGNORECASE,
)
RE_FOLLOWING_TABLE = re.compile(r"[Tt]he following table[^.]+\.", re.IGNORECASE)
RE_ORPHAN_NUMS = re.compile(
    r"(?<!\w)(?:{val})(?:\s+(?:{val})){{0,3}}(?!\s*\w{{4}})".format(val=_VAL),
    re.IGNORECASE,
)
RE_PAGE_NUM = re.compile(r"\b\d{1,3}\b(?=\s+[A-Z])")

RE_TABLE_INTRO = re.compile(
    r"(?:[Tt]?he\s+following\s+(?:table|chart|graph|diagram|schedule|summary)"
    r"|[Tt]?he\s+table\s+below"
    r"|set\s+forth\s+below\s+is"
    r"|below\s+is\s+a\s+summary"
    r"|[Tt]?he\s+following\s+(?:is\s+a\s+)?(?:reconciliation|summary|illustrat)"
    r"|(?:a\s+summary|[Tt]?he\s+components?)\s+(?:of\s+\w+\s+)?(?:is|was|were|are)\s+as\s+follows"
    r"|(?:revenue|sales|expenses?|income|results|costs?)\s+(?:by\s+\w+\s+)?(?:is|was|were|are)\s+as\s+follows"
    r")[^.:]*[.:]",
    re.IGNORECASE,
)
RE_TABLE_LABELS = re.compile(
    r"\((?:In\s+)?(?:thousands|millions|billions)(?:\s*,\s*except[^)]+)?\)"
    r"\((?:In\s+)?(?:thousand|million|billion)(?:\s*,\s*except[^)]+)?\)"
    r"|\((?:Unaudited|Audited)\)"
    r"|\(As\s+a\s+Percentage\s+of[^)]+\)",
    re.IGNORECASE,
)
RE_COLUMN_JUNK = re.compile(
    r"(?:[\s\$%\(\)\-,]|Variance|Amount|Change|Total){3,}", re.IGNORECASE
)

# ── 1C. Boilerplate sentence patterns ──────────────────────────────────────
S = r"[\s,/&]+"

BOILERPLATE_SENTENCE_PATTERNS = [
    # Temporal / period markers
    re.compile(rf"(?:fiscal|calendar){S}(?:year|period){S}(?:ended?|ending)", re.I),
    re.compile(
        rf"(?:first|second|third|fourth){S}(?:quarter|year|half|nine\s+months)", re.I
    ),
    re.compile(
        rf"(?:during|for){S}(?:the|a){S}(?:first|second|third|fourth){S}(?:quarter|year)",
        re.I,
    ),
    re.compile(rf"nine{S}months{S}ended?", re.I),
    re.compile(rf"(?:first|second){S}half{S}(?:of|the)", re.I),
    re.compile(rf"fiscal{S}(?:year|period){S}(?:20\d{{2}}|ended)", re.I),
    re.compile(
        rf"(?:year|first_year|second_year|third_year|fourth_year){S}to{S}date", re.I
    ),
    re.compile(rf"(?:current|prior){S}(?:fiscal|calendar){S}(?:year|period)", re.I),
    re.compile(
        rf"(?:same|corresponding){S}(?:period|month|year){S}(?:of|in)?{S}(?:the|prior)",
        re.I,
    ),
    # Cross-reference boilerplate
    re.compile(
        rf"(?:please{{S}})?(?:see|refer){S}(?:to{S})?note[s]?{S}(?:\d+{S})?(?:to{S})?(?:the{S})?(?:condensed{S})?(?:consolidated{S})?financial",
        re.I,
    ),
    re.compile(rf"see{S}accompanying{S}notes", re.I),
    re.compile(rf"read{S}(?:in{S})?conjunction{S}with", re.I),
    re.compile(
        rf"(?:included|appearing){S}(?:elsewhere{S})?in{S}(?:this|the|our){S}(?:quarterly|annual){S}report",
        re.I,
    ),
    re.compile(rf"(?:form|filed){S}(?:on{S})?(?:form{S})?10-[kq]", re.I),
    re.compile(
        rf"(?:please|note|refer){S}to{S}(?:the|our)?{S}(?:discussion|additional(?:{S}information)?)",
        re.I,
    ),
    re.compile(
        rf"(?:for|in){S}(?:further|additional){S}(?:discussion|information|details?)",
        re.I,
    ),
    re.compile(
        rf"(?:see|refer|please{S}see){S}(?:the|our){S}(?:discussion|section){S}(?:entitled|titled)",
        re.I,
    ),
    re.compile(
        rf"(?:for{S})?(?:recently{S})?(?:issued|adopted){S}accounting{S}(?:standards?|pronouncements?)",
        re.I,
    ),
    re.compile(
        rf"(?:discussed|discussed{S}in|described{S}in){S}(?:note|section)", re.I
    ),
    re.compile(
        rf"additional{S}(?:discussion|information|details?){S}(?:can|may){S}be{S}found",
        re.I,
    ),
    # Generic quantitative terms
    re.compile(r"\b(?:a[\s\.\,]*)?table[\s\.\,]*of[\s\.\,]*contents\b", re.I),
    re.compile(
        rf"(?:in{S})?(?:millions?|billions?|thousands?){S}(?:of|dollars?)?", re.I
    ),
    re.compile(rf"(?:approximately|roughly){S}(?:$|\d)", re.I),
    re.compile(rf"(?:total|net|gross){S}(?:revenue|sales|expenses?|amount)", re.I),
    re.compile(
        rf"(?:respectively|in{S}the{S}same|comparable){S}(?:period|amount)", re.I
    ),
    re.compile(
        rf"(?:dollar|amount){S}(?:amounts?|in{S}thousands?|in{S}millions?)", re.I
    ),
    re.compile(
        rf"(?:increased|decreased|improved|declined){S}(?:to|by)?{S}(?:approximately|\$|(?:\d+))",
        re.I,
    ),
    # Generic functional phrases
    re.compile(
        rf"(?:may|could|would|should|might){S}(?:not{S})?(?:be|have){S}(?:affected|impacted|changed)",
        re.I,
    ),
    re.compile(
        rf"(?:due{S})?to{S}(?:certain|various|several){S}(?:factors?|reasons)", re.I
    ),
    re.compile(
        rf"(?:based{S})?(?:on|upon){S}(?:our|the|current){S}(?:estimates?|assumptions?|judgments?)",
        re.I,
    ),
    re.compile(rf"(?:is|was|were){S}(?:also{S})?(?:new|newly|currently)", re.I),
    re.compile(
        rf"(?:using|used{S})?(?:new|updated){S}(?:methodology|method|approach)", re.I
    ),
    re.compile(rf"(?:as{S})?(?:well{S})?as{S}(?:our|the)", re.I),
    re.compile(rf"(?:for{S})?us(?:{S}to|{S}(?:and|the))", re.I),
    re.compile(rf"(?:currently|presently){S}(?:does|is|are)", re.I),
    re.compile(rf"(?:could{S})?affect{S}(?:our|the){S}(?:results?|operations?)", re.I),
    re.compile(rf"in{S}(?:addition|order|part)", re.I),
    # Geographic / jurisdiction boilerplate
    re.compile(
        rf"(?:united{S}states?|u\.?s\.?|outside{S}(?:the|our)?{S}(?:united{S})?states?)",
        re.I,
    ),
    re.compile(
        rf"(?:domestic|international|foreign|non-?us){S}(?:operations?|revenue|sales)",
        re.I,
    ),
    re.compile(
        rf"(?:in|outside|within){S}(?:the|our)?{S}(?:united{S}states?|u\.s\.)", re.I
    ),
    re.compile(
        rf"(?:united{S}states?|foreign){S}operations?{S}(?:represent|comprise)", re.I
    ),
    re.compile(
        rf"(?:foreign{S})?exchange{S}(?:rates?|fluctuations?|gains?|losses?)", re.I
    ),
    re.compile(
        rf"(?:geographic|jurisdictional){S}mix{S}of{S}(?:revenues?|sales)", re.I
    ),
    # Company metadata (HQ, website, address)
    re.compile(
        rf"(?:headquarters?|head\s*office){S}(?:is|are|located|based)?{S}", re.I
    ),
    re.compile(
        rf"(?:headquartered?|head\s*office){S}(?:in|are|located|based)?{S}", re.I
    ),
    re.compile(
        rf"(?:located|based){S}(?:in|at){S}.*(?:headquarters?|head\s*office)", re.I
    ),
    re.compile(rf"(?:website|web\s*site){S}(?:is|at|located)?{S}", re.I),
    re.compile(r"https?://\S+", re.I),
    re.compile(r"www\.\S+", re.I),
    re.compile(rf"investor{S}relations{S}(?:website|page|site)?", re.I),
    re.compile(rf"(?:visit|see){S}(?:our{S})?(?:website|web\s*site)", re.I),
    re.compile(
        rf"\d+{S}.*(?:street|st\.?|avenue|ave\.?|road|rd\.?|blvd|drive|dr\.?)", re.I
    ),
    re.compile(rf"(?:city|state|zip|postal){S}(?:code)?", re.I),
    # Forward-looking / safe harbor stragglers
    re.compile(rf"forward[_-]?looking{S}statements?", re.I),
    re.compile(
        rf"actual{S}results?{S}(?:could|may|might|will){S}(?:differ|be){S}materially",
        re.I,
    ),
    re.compile(rf"(?:should|do){S}not{S}place{S}undue{S}reliance", re.I),
    re.compile(rf"(?:no|assume{S}no){S}obligation{S}(?:to{S})?update", re.I),
    re.compile(rf"(?:known|unknown){S}risks?{S}uncertainties", re.I),
    re.compile(rf"safe{S}harbor", re.I),
    re.compile(rf"private{S}securities{S}litigation{S}reform", re.I),
    # Accounting policy boilerplate
    re.compile(
        rf"critical{S}accounting{S}polic(?:y|ies){S}(?:and{S})?estimates?", re.I
    ),
    re.compile(rf"(?:significant|critical){S}accounting{S}polic(?:y|ies)", re.I),
    re.compile(
        rf"managements?{S}discussion{S}(?:and{S})?analysis{S}(?:of{S})?financial{S}condition",
        re.I,
    ),
    re.compile(
        rf"preparation{S}(?:of{S})?financial{S}(?:statements?{S})?.*requires{S}(?:us|management){S}(?:to{S})?make{S}estimates",
        re.I,
    ),
    re.compile(
        rf"actual{S}results{S}could{S}differ{S}from{S}(?:those{S})?estimates", re.I
    ),
    re.compile(rf"generally{S}accepted{S}accounting{S}principles", re.I),
    re.compile(
        rf"(?:management|we){S}discussed{S}(?:the{S})?development{S}(?:and{S})?selection{S}(?:of{S})?critical",
        re.I,
    ),
    re.compile(
        rf"audit{S}committee{S}(?:has{S})?reviewed{S}(?:the{S})?disclosures", re.I
    ),
    # Fair value hierarchy
    re.compile(rf"fair{S}value{S}hierarchy{S}prioriti[sz]es", re.I),
    re.compile(rf"level{S}[123]{S}(?:inputs|consists?|assets?|measurements?)", re.I),
    re.compile(
        rf"quoted{S}(?:market{S})?prices{S}.*active{S}markets?{S}(?:for{S})?identical",
        re.I,
    ),
    re.compile(
        rf"unobservable{S}inputs{S}(?:for{S}the{S}asset|that{S}are{S}supported)", re.I
    ),
    # Tax boilerplate
    re.compile(
        rf"(?:full{S})?valuation{S}allowance{S}(?:against|on){S}.*deferred{S}tax", re.I
    ),
    # Goodwill & impairment
    re.compile(
        rf"goodwill{S}(?:is{S})?subject{S}(?:to{S})?(?:an{S})?(?:annual{S})?impairment{S}test",
        re.I,
    ),
    re.compile(rf"other-?than-?temporar(?:y|ily){S}impair(?:ment|ed)", re.I),
    # Investment policy
    re.compile(
        rf"available-?for-?sale{S}(?:investments?|securities){S}(?:are{S})?subject{S}(?:to{S})?(?:a{S})?periodic{S}impairment",
        re.I,
    ),
    re.compile(
        rf"securities{S}(?:are{S})?(?:reported|carried|measured){S}(?:at{S})?fair{S}value",
        re.I,
    ),
    # Stock-based compensation
    re.compile(rf"black-?scholes{S}(?:option{S})?pricing{S}model", re.I),
    re.compile(
        rf"forfeitures{S}(?:are{S})?estimated{S}(?:based{S})?(?:on|upon){S}historical{S}experience",
        re.I,
    ),
    # Litigation
    re.compile(rf"(?:aggressively|vigorously){S}defending{S}.*litigation", re.I),
    re.compile(
        rf"legal{S}(?:proceedings|matters){S}(?:are{S})?inherently{S}unpredictable",
        re.I,
    ),
    re.compile(
        rf"(?:we{S})?record{S}(?:a{S})?(?:provision|liability).*(?:when{S})?.*(?:both{S})?probable",
        re.I,
    ),
    # Liquidity boilerplate
    re.compile(
        rf"(?:we{S})?believe{S}(?:that{S})?(?:our{S})?existing{S}(?:cash|liquidity).*(?:sufficient|adequate)",
        re.I,
    ),
    re.compile(
        rf"additional{S}(?:funds|financing){S}may{S}not{S}(?:be{S})?available{S}(?:on{S})?(?:favorable{S})?terms",
        re.I,
    ),
    re.compile(
        rf"(?:next{S})?(?:at{S}least{S})?(?:the{S})?next{S}(?:12|twelve){S}months", re.I
    ),
    # Stock repurchase
    re.compile(
        rf"repurchases{S}(?:may{S}be{S})?made{S}(?:in{S}the{S})?open{S}market", re.I
    ),
    re.compile(
        rf"(?:the{S})?program{S}(?:does{S})?not{S}obligate{S}.*common{S}stock", re.I
    ),
    # Off-balance sheet
    re.compile(
        rf"(?:do|did){S}not{S}have{S}(?:any{S})?(?:material{S})?off-?balance{S}sheet",
        re.I,
    ),
    re.compile(rf"(?:structured{S}finance{S}or{S})?special{S}purpose{S}entities", re.I),
    re.compile(rf"unconsolidated{S}(?:entities|organizations)", re.I),
    # Contractual obligations
    re.compile(
        rf"(?:no{S})?material{S}changes{S}(?:in{S})?.*contractual{S}obligations", re.I
    ),
    # Covenant compliance
    re.compile(
        rf"(?:was|were|is|are){S}in{S}compliance{S}with{S}all{S}(?:financial{S})?covenants",
        re.I,
    ),
    re.compile(rf"customary{S}(?:negative{S})?covenants{S}(?:that{S})?limit", re.I),
    # Geographic / segment disclosure
    re.compile(
        rf"revenue{S}(?:by{S})?geographic{S}(?:region|location){S}(?:is{S})?(?:based|allocated)",
        re.I,
    ),
    # Risk factors reference
    re.compile(
        rf"(?:discussed|described){S}(?:in{S})?(?:the{S})?(?:section{S})?(?:entitled|titled){S}risk{S}factors",
        re.I,
    ),
    # Table / numeric artifacts
    re.compile(r"^[\d\s,.$%()]+$"),
    re.compile(rf"^(?:dollars{S}(?:in{S})?(?:thousands|millions|billions))$", re.I),
    re.compile(rf"^(?:dollar{S}(?:in{S})?(?:thousand|million|billion))$", re.I),
    re.compile(rf"^(?:in{S}thousands{S}except{S}per{S}share)", re.I),
    re.compile(rf"^(?:unaudited)$", re.I),
    # Insurance-specific boilerplate
    re.compile(rf"(?:a{S})?combined{S}ratio{S}(?:of{S})?under{S}100", re.I),
    re.compile(
        rf"property{S}(?:and{S})?(?:/)?casualty{S}(?:insurance{S})?industry{S}(?:is{S})?cyclical",
        re.I,
    ),
]

# ── Structural noise patterns (used in clean_text) ──────────────────────────
RE_ITEM_TAG = re.compile(r"\bItem\s+\d+[A-Z]?\.\s*")
RE_PART_TAG = re.compile(r"\bPart\s+[IVX]+,?\s*")
RE_COPYRIGHT = re.compile(r"©?\s*\d{4}\s+\w[^.]*(?:Corporation|Inc|LLC|Ltd)[^.]*\.")
RE_WHITESPACE = re.compile(r"\s{2,}")


## Helper Functions


In [3]:
def parse_filename(filename: str) -> dict:
    """Parse MDA filename: CompanyName_10-K/Q_YYYY-MM-DD_MDA.txt"""
    m = re.match(
        r"^(?P<company>.+)_(?P<ftype>10-[KQ])_(?P<year>\d{4})-(?P<month>\d{2})-(?P<day>\d{2})(?:T[\d_]+)?_(?P<section>.+)",
        Path(filename).stem,
    )
    if not m:
        return None
    month = int(m.group("month"))
    return {
        "company_name": m.group("company"),
        "filing_type": m.group("ftype"),
        "filing_date": f"{m.group('year')}-{m.group('month')}-{m.group('day')}",
        "year": m.group("year"),
        "quarter": f"Q{(month - 1) // 3 + 1}",
    }


def strip_fls(raw: str) -> str:
    """Step 1A: Remove forward-looking boilerplate before the first substantive heading."""
    m = _FLS_PATTERN.search(raw)
    return raw[m.start() :] if m else raw


def remove_tables(text: str) -> str:
    """Step 1B: Strip financial table lines; leave prose unchanged."""
    # Filings rendered as a single long line have no table structure to strip.
    if text.count("\n") < 20:
        return text

    def is_table_line(line: str) -> bool:
        s = line.strip()
        if not s:
            return False
        if len(s) < 220 and (RE_HEADER_ROW.search(s) or RE_TABLE_LABELS.search(s)):
            return True
        numeric_hits = len(re.findall(_VAL, s))
        if numeric_hits >= 3 and len(s.split()) <= 30:
            return True
        alpha = sum(ch.isalpha() for ch in s)
        if alpha == 0:
            return True
        if (
            (1.0 - alpha / max(len(s), 1)) > 0.75
            and numeric_hits >= 2
            and len(s.split()) <= 40
        ):
            return True
        return False

    return "\n".join(line for line in text.splitlines() if not is_table_line(line))


def is_boilerplate_sentence(sentence: str) -> bool:
    """Step 1C: True if the sentence matches a known boilerplate pattern."""
    return any(p.search(sentence) for p in BOILERPLATE_SENTENCE_PATTERNS)


def clean_text(text: str) -> str:
    """Step 2: Lowercase, strip structural noise, mask numbers, normalize whitespace."""
    text = RE_ITEM_TAG.sub(" ", text)
    text = RE_PART_TAG.sub(" ", text)
    text = RE_COPYRIGHT.sub(" ", text)
    text = text.lower()
    text = re.sub(r"\$[\d,\.]+", " NUM ", text)  # dollar amounts → NUM
    text = re.sub(r"(?<!\w)[\d,\.]+%", " NUM ", text)  # percentages → NUM
    text = re.sub(r"(?<!\w)\d[\d,\.]*(?!\w)", " NUM ", text)  # standalone numbers → NUM
    text = re.sub(r"[^\w\s\.\!\?\-\']", " ", text)  # drop non-sentence punctuation
    return RE_WHITESPACE.sub(" ", text).strip()


# ── Fragment / cut-off detection ─────────────────────────────────────────────

# Sentence ends with these → likely truncated mid-sentence
_CUTOFF_LAST_TOKENS = {
    "a",
    "an",
    "the",
    "and",
    "or",
    "but",
    "to",
    "of",
    "in",
    "on",
    "at",
    "by",
    "for",
    "from",
    "with",
    "as",
    "than",
    "is",
    "are",
    "was",
    "were",
    "be",
    "been",
    "being",
    "we",
    "our",
    "their",
    "its",
    "this",
    "that",
    "these",
    "those",
    "have",
    "has",
    "had",
    "will",
    "would",
    "can",
    "could",
    "should",
    "may",
    "might",
}

# Sentence ends with one of these phrases → dangling cross-reference
_CUTOFF_END_PHRASES = (
    "included in",
    "set forth in",
    "described in",
    "provided in",
    "presented in",
    "disclosed in",
    "as follows",
    "would be",
    "could be",
    "may be",
    "than we can",
)

# Sentence starts with these → fragment split from a longer sentence
_FRAGMENT_FIRST_TOKENS = {
    "which",  # relative clause:       "which consisted of..."
    "and",  # coordinating conj.:    "and revenues were..."
    "or",  # coordinating conj.:    "or the equivalent..."
    "as",  # subordinating / prep.: "as compared to..." / "as a result..."
}

# Sentence starts with these 2-word phrases → dangling prepositional fragment
_FRAGMENT_FIRST_PHRASES = ("of the", "of our", "of their", "of its")


def _strip_punct(tok: str) -> str:
    """Strip non-alpha characters from a token (used for first/last word checks)."""
    return re.sub(r"[^a-z]", "", tok.lower())


def is_likely_cutoff_sentence(sentence: str) -> bool:
    """Return True if the sentence looks like a fragment or mid-cut truncation."""
    s = sentence.strip().lower()
    words = s.split()
    if len(words) < 8:
        return False

    # Dangling ending phrases (cross-references with a missing target)
    if any(s.endswith(p) or s.endswith(p + ".") for p in _CUTOFF_END_PHRASES):
        return True

    # Ends with a connector/function word → cut before the sentence finished
    if _strip_punct(words[-1]) in _CUTOFF_LAST_TOKENS:
        return True

    # Mid-thought punctuation at end
    if s.endswith((",", ":", ";", "-", "/")):
        return True

    # Starts with a fragment-starting token (relative clause / conjunction)
    if _strip_punct(words[0]) in _FRAGMENT_FIRST_TOKENS:
        return True

    # Starts with a dangling prepositional phrase ("of the ...", "of our ...")
    if any(" ".join(words[:2]).startswith(p) for p in _FRAGMENT_FIRST_PHRASES):
        return True

    return False


def segment_sentences(text: str) -> list[str]:
    """Step 3: Split, filter fragments and boilerplate, deduplicate."""
    seen = set()
    result = []
    for s in sent_tokenize(text):
        s = s.strip()
        if len(s.split()) < MIN_SENT_WORDS:
            continue
        if is_boilerplate_sentence(s) or is_likely_cutoff_sentence(s):
            continue
        key = RE_WHITESPACE.sub(" ", s).strip().lower()
        if key in seen:
            continue
        seen.add(key)
        result.append(s)
    return result


def process_one_file(fp: Path) -> dict | None:
    """Process a single MDA file — runs in a joblib worker process."""
    meta = parse_filename(fp.name)
    if meta is None:
        return {"_skip": "bad_filename", "_name": fp.name}

    text = fp.read_text(encoding="utf-8", errors="replace")
    text = strip_fls(text)
    text = remove_tables(text)
    text = clean_text(text)
    sentences = segment_sentences(text)

    if not sentences:
        return {"_skip": "empty", "_name": fp.name}

    return {
        "doc_id": f"{meta['company_name']}_{meta['filing_type']}_{meta['filing_date']}",
        "company_name": meta["company_name"],
        "filing_type": meta["filing_type"],
        "filing_date": meta["filing_date"],
        "year": meta["year"],
        "quarter": meta["quarter"],
        "sentences": sentences,
        "clean_text": " ".join(sentences),
        "n_sentences": len(sentences),
    }


## Main Pipeline

1. **Section-level removal** — FLS / safe harbor, tables, headers, boilerplate paragraphs
2. **Text cleaning** — lowercase, strip `$/%`, normalize whitespace
3. **Sentence segmentation** — split + drop fragments < 10 words


In [4]:
def run_preprocessing(mda_folder: str, file_pattern: str) -> pd.DataFrame:
    """
    Shared preprocessing pipeline (Steps 1–3), parallelised with joblib.

    Returns:
        df_shared — one row per filing with:
            sentences   (list[str])  → for Sentiment team
            clean_text  (str)        → for Topic team
    """
    mda_path = Path(mda_folder)
    files = sorted(mda_path.glob(file_pattern))
    print(f"✅ Found {len(files)} MDA files in {mda_folder}")

    if not files:
        raise FileNotFoundError(f"No files matching '{file_pattern}' in {mda_folder}")

    results = Parallel(n_jobs=N_JOBS, backend="loky", verbose=5)(
        delayed(process_one_file)(fp) for fp in files
    )

    rows = []
    skipped_meta = 0
    skipped_empty = 0

    for r in results:
        if r is None:
            skipped_meta += 1
        elif "_skip" in r:
            if r["_skip"] == "bad_filename":
                print(f"  SKIPPED (bad filename): {r['_name']}")
                skipped_meta += 1
            else:
                print(f"  SKIPPED (no sentences after cleaning): {r['_name']}")
                skipped_empty += 1
        else:
            rows.append(r)

    df = pd.DataFrame(rows)
    df = df.sort_values(["company_name", "filing_date"]).reset_index(drop=True)

    print(f"\n✅ Done!")
    print(f"✅ Processed:  {len(rows)} documents")
    print(
        f"✅ Skipped:    {skipped_meta} (bad filename) + {skipped_empty} (empty after cleaning)"
    )
    print(f"✅ Output:     {df.shape[0]} rows × {df.shape[1]} columns")
    return df


## Run & Export


In [5]:
df_shared = run_preprocessing(MDA_FOLDER, FILE_PATTERN)
# df_shared = run_preprocessing("./sample", FILE_PATTERN)

# ── Quick inspection ───────────────────────────────────────────────
print("\n=== df_shared (first 5 rows) ===")
print(
    df_shared[["doc_id", "company_name", "filing_type", "filing_date", "n_sentences"]]
    .head(5)
    .to_string(index=False)
)

# ── Coverage summary ───────────────────────────────────────────────
print(f"\n=== Summary ===")
print(f"Total documents:      {len(df_shared)}")
print(f"Total sentences:      {df_shared['n_sentences'].sum():,}")
print(f"Avg sentences/doc:    {df_shared['n_sentences'].mean():.1f}")
print(f"Companies:            {df_shared['company_name'].nunique()}")

print("\n=== Documents per company (top 10) ===")
print(
    df_shared.groupby("company_name")["doc_id"]
    .count()
    .sort_values(ascending=False)
    .head(10)
    .to_string()
)

✅ Found 18183 MDA files in datasets/raw_mda/


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 11 concurrent workers.
[Parallel(n_jobs=-1)]: Done  50 tasks      | elapsed:    3.0s
[Parallel(n_jobs=-1)]: Done 438 tasks      | elapsed:    5.5s
[Parallel(n_jobs=-1)]: Done 1446 tasks      | elapsed:   11.1s
[Parallel(n_jobs=-1)]: Done 2742 tasks      | elapsed:   18.9s
[Parallel(n_jobs=-1)]: Done 4326 tasks      | elapsed:   29.0s
[Parallel(n_jobs=-1)]: Done 6198 tasks      | elapsed:   41.4s
[Parallel(n_jobs=-1)]: Done 8358 tasks      | elapsed:   55.4s
[Parallel(n_jobs=-1)]: Done 10806 tasks      | elapsed:  1.2min
[Parallel(n_jobs=-1)]: Done 13542 tasks      | elapsed:  1.4min
[Parallel(n_jobs=-1)]: Done 16566 tasks      | elapsed:  1.7min


  SKIPPED (no sentences after cleaning): Aehr_Test_Systems_10-Q_2020-01-14_MDA_MDA.txt
  SKIPPED (no sentences after cleaning): Aehr_Test_Systems_10-Q_2020-04-14_MDA_MDA.txt
  SKIPPED (no sentences after cleaning): Aehr_Test_Systems_10-Q_2020-10-14_MDA_MDA.txt
  SKIPPED (no sentences after cleaning): Aehr_Test_Systems_10-Q_2021-01-14_MDA_MDA.txt
  SKIPPED (no sentences after cleaning): Aehr_Test_Systems_10-Q_2021-04-13_MDA_MDA.txt
  SKIPPED (no sentences after cleaning): HP_10-Q_2012-09-10_MDA.txt
  SKIPPED (no sentences after cleaning): HP_10-Q_2013-06-06_MDA.txt
  SKIPPED (no sentences after cleaning): HP_10-Q_2013-09-09_MDA.txt
  SKIPPED (no sentences after cleaning): HP_10-Q_2014-03-11_MDA.txt
  SKIPPED (no sentences after cleaning): HP_10-Q_2014-06-09_MDA.txt
  SKIPPED (no sentences after cleaning): HP_10-Q_2014-09-09_MDA.txt
  SKIPPED (no sentences after cleaning): HP_10-Q_2015-03-11_MDA.txt
  SKIPPED (no sentences after cleaning): HP_10-Q_2015-06-08_MDA.txt
  SKIPPED (no sentenc

[Parallel(n_jobs=-1)]: Done 18183 out of 18183 | elapsed:  1.9min finished


## Export


In [6]:
# Audit: flag sentences that look like fragments or cut-offs in the processed output.
flagged = []
total_sentences = 0
for _, row in df_shared.iterrows():
    sents = row.get("sentences", [])
    if not isinstance(sents, list):
        continue
    total_sentences += len(sents)
    for sent in sents:
        if is_likely_cutoff_sentence(sent):
            flagged.append(
                {
                    "doc_id": row["doc_id"],
                    "company_name": row["company_name"],
                    "filing_date": row["filing_date"],
                    "sentence": sent,
                }
            )

df_midcut = pd.DataFrame(flagged)
unique_midcut = (
    df_midcut.drop_duplicates(subset=["sentence"]) if not df_midcut.empty else df_midcut
)
rate = len(df_midcut) / total_sentences * 100 if total_sentences else 0.0

print(f"Total sentences audited:          {total_sentences:,}")
print(f"Likely fragments / cut-offs:      {len(df_midcut):,}")
print(f"Unique likely fragments:          {len(unique_midcut):,}")
print(f"Fragment rate:                    {rate:.3f}%")

if not unique_midcut.empty:
    print("\nFirst 10 examples:")
    for i, (_, r) in enumerate(unique_midcut.head(10).iterrows(), start=1):
        print(f"{i}. [{r['doc_id']}] {r['sentence']}")
else:
    print("\nNo fragments detected.")

if not df_midcut.empty:
    Path("datasets/interim").mkdir(parents=True, exist_ok=True)
    df_midcut.to_parquet("datasets/interim/cutoff_sentences_detected.parquet", index=False)
    print(f"\nSaved to datasets/interim/cutoff_sentences_detected.parquet")


Total sentences audited:          1,251,937
Likely fragments / cut-offs:      0
Unique likely fragments:          0
Fragment rate:                    0.000%

No fragments detected.


In [7]:
OUTPUT_DIR = Path("datasets/final")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PARQUET_PATH = OUTPUT_DIR / "mda_shared_preprocessed.parquet"

df_shared.to_parquet(PARQUET_PATH, index=False)
print(f"✅ Saved: {PARQUET_PATH}")


✅ Saved: datasets/final/mda_shared_preprocessed.parquet
